In [ ]:
import ee
import json
import os

try:
    ee.Initialize(project='urban-heat-island-482910')
except Exception as e:
    print("Earth Engine authentication failed. Run 'earthengine authenticate'.")
    raise

In [ ]:
def get_region_of_interest(geojson_path):
    with open(geojson_path) as f:
        data = json.load(f)
    geom = data['features'][0]['geometry']
    return ee.Geometry(geom)

geojson_path = 'export.geojson'
roi = get_region_of_interest(geojson_path)

In [ ]:
def mask_s2_clouds(image):
    cloud_prob = ee.Image(image.get('cloud_probability')).select('probability')
    return image.updateMask(cloud_prob.lt(50)).divide(10000)

def add_indices(image):
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return image.addBands([ndbi, ndvi, mndwi])


In [ ]:
def mask_l8_clouds(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 0).eq(0) & qa.bitwiseAnd(1 << 3).eq(0) & qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(mask)

def get_lst(image):
    st = image.select('ST_B10').updateMask(image.select('ST_B10').gt(0))
    st_kelvin = st.multiply(0.00341802).add(149.0)
    return image.addBands(st_kelvin.subtract(273.15).rename('LST'))


In [ ]:
elevation = ee.Image("NASA/NASADEM_HGT/001").select('elevation').clip(roi)
slope = ee.Terrain.slope(elevation).rename('slope')

proj = ee.Projection("EPSG:32748").atScale(10)
roi_context = roi.buffer(100000)
worldcover = ee.Image("ESA/WorldCover/v200/2021").select("Map").clip(roi_context).reproject(proj)
water = worldcover.eq(80)
land = water.Not()
coast = land.focal_min(1).neq(land).selfMask()
pixel_size = worldcover.projection().nominalScale()
dist_coast = coast.fastDistanceTransform(256).sqrt().multiply(pixel_size).rename("dist_coast").updateMask(land).clip(roi)


In [ ]:
def make_grid(roi, cell_size=750):
    proj = ee.Projection("EPSG:32748")
    return roi.transform(proj, 1).coveringGrid(proj.atScale(cell_size))


In [ ]:
def process_year(year, roi, sample_size=10000):
    start_date = f'{year}-05-01'
    end_date = f'{year}-09-30'

    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
    s2_cloud_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(roi).filterDate(start_date, end_date)

    s2_with_cloud = ee.Join.saveFirst('cloud_probability').apply(primary=s2, secondary=s2_cloud_prob, condition=ee.Filter.equals('system:index', 'system:index'))
    s2_median = ee.ImageCollection(s2_with_cloud).map(mask_s2_clouds).median().clip(roi)
    s2_final = add_indices(s2_median).select(['NDVI','NDBI','MNDWI'])

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    if year >= 2022:
        l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        l8 = l8.merge(l9)
    l8_lst = l8.filterBounds(roi).filterDate(start_date, end_date).filter(ee.Filter.lt('CLOUD_COVER',50)).map(mask_l8_clouds).map(get_lst).median().select('LST').clip(roi)

    stack = s2_final.addBands([l8_lst, elevation, slope, dist_coast])
    stack_with_year = stack.addBands(ee.Image.constant(year).rename('year'))

    samples = stack_with_year.sample(region=roi, scale=10, numPixels=sample_size, geometries=True)

    grid = make_grid(roi)
    grid = grid.map(lambda f: f.set('grid_id', f.id()))
    samples_grid = ee.Join.saveFirst('grid').apply(primary=samples, secondary=grid, condition=ee.Filter.intersects('.geo','.geo'))
    samples_final = samples_grid.map(lambda f: f.set('grid_id', ee.Feature(f.get('grid')).get('grid_id')))

    return samples_final


In [ ]:
years = [2019, 2022, 2025]
collections = []

def add_coords(feat):
    coords = feat.geometry().coordinates()
    return feat.set({'longitude': coords.get(0), 'latitude': coords.get(1)})

for y in years:
    try:
        fc = process_year(y, roi)
        fc = fc.map(add_coords)
        collections.append(fc)
        print(f"Prepared {y}")
    except Exception as e:
        print(f"Error for {y}: {e}")

if collections:
    merged_fc = ee.FeatureCollection(collections).flatten()
    task = ee.batch.Export.table.toDrive(collection=merged_fc, description='uhi_data_multiyear_export', fileFormat='CSV')
    task.start()
    print(f"Export task started: {task.id}")
